# 01 · Preparing the data
**Lighting the Way? Street light availability and street crime across urban Northern Ireland**

This notebook cleans the crime and street-light data, counts them in NISRA's
**1 km grid squares**, and selects the urban squares we will study.

Download the data into `data/raw/` first (see `data/README.md`). Run the cells from the top.

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## Section 1 — Crime

data.police.uk gives one CSV per month. `rglob()` finds all twelve files, and
`pd.concat()` stacks them into one table.

In [ ]:
crime_files = sorted(Path('../data/raw/police').rglob('*-street.csv'))
crime = pd.concat([pd.read_csv(f) for f in crime_files])
crime.shape

In [ ]:
crime['Crime type'].value_counts()

Anti-social behaviour is recorded as incidents, not crimes, so we remove it.
Then check for missing values.

In [ ]:
crime = crime[crime['Crime type'] != 'Anti-social behaviour']
crime.isna().sum()

Rows without coordinates cannot be placed on the map, so we drop them.

PSNI reuses the same Crime ID in different months for different crimes, so
duplicates must be checked on **Crime ID and Month together**.

In [ ]:
crime = crime.dropna(subset=['Longitude', 'Latitude'])
crime = crime.drop_duplicates(subset=['Crime ID', 'Month'])
crime.shape

### Two crime groups

**Street crime** happens outdoors, mostly between strangers, where being seen
after dark plausibly matters. The **comparison** group mostly happens indoors
or between people who know each other, where street lighting should not matter.

In [ ]:
street_types = ['Robbery', 'Theft from the person', 'Vehicle crime',
                'Criminal damage and arson']
comparison_types = ['Violence and sexual offences', 'Shoplifting']

crime['group'] = 'other'
crime.loc[crime['Crime type'].isin(street_types), 'group'] = 'street'
crime.loc[crime['Crime type'].isin(comparison_types), 'group'] = 'comparison'
crime['group'].value_counts()

Create points from longitude and latitude. `to_crs()` changes their projection
to EPSG:29902, the Irish Grid used by the NISRA grid squares (metres).

In [ ]:
crime_map = gpd.GeoDataFrame(
    crime,
    geometry=gpd.points_from_xy(crime.Longitude, crime.Latitude),
    crs='EPSG:4326'
)
crime_map = crime_map.to_crs(epsg=29902)

## Section 2 — Street lights

The DfI register lists every public street-light column. A few rows have
impossible coordinates (0, or latitude/longitude typed into the easting
column), so keep only eastings and northings inside Northern Ireland.

In [ ]:
lights = pd.read_csv('../data/raw/streetlights/Street Lighting Assets.csv', low_memory=False)
lights.shape

In [ ]:
lights = lights[lights['EASTING'].between(180000, 370000) &
                lights['NORTHING'].between(300000, 460000)]
lights = lights.drop_duplicates(subset=['EASTING', 'NORTHING'])
lights.shape

Several lanterns can share one column, so duplicate locations are counted once.

Lantern wattage is only recorded for about 12% of lights, so we record lamp
type instead. `str.contains('LED')` marks the LED lanterns.

In [ ]:
lights['LED'] = lights['LANTERN_BOQ_NAME'].str.upper().str.contains('LED', na=False)
lights['LED'].mean()

In [ ]:
lights_map = gpd.GeoDataFrame(
    lights,
    geometry=gpd.points_from_xy(lights.EASTING, lights.NORTHING),
    crs='EPSG:29902'
)

## Section 3 — The 1 km grid and population

Read the grid squares and join the Census 2021 population. `PERSONS` counts
usual residents. NISRA hides the population of very small squares (shown as 0).

In [ ]:
grid = gpd.read_file('../data/raw/nisra_grid/Grid_Square_2021_1km.shp')
grid = grid.to_crs(epsg=29902)
grid = grid.rename(columns={'GSQ21_1km': 'cell_id'})[['cell_id', 'geometry']]

census = pd.read_csv('../data/raw/nisra_grid/census-2021-grid-square-product-1km-full-dataset.csv',
                     usecols=['Geography', 'PERSONS'])
census = census.rename(columns={'Geography': 'cell_id', 'PERSONS': 'population'})

grid = grid.merge(census, on='cell_id', how='left')
grid['population'] = grid['population'].fillna(0)
grid.shape

## Section 4 — Count crimes and lights in each square

`sjoin()` attaches the `cell_id` of the square each point falls in. A point
exactly on a border touches two squares, so we keep its first match only.

In [ ]:
crime_in_grid = gpd.sjoin(crime_map, grid[['cell_id', 'geometry']], predicate='intersects')
crime_in_grid = crime_in_grid[~crime_in_grid.index.duplicated()]

lights_in_grid = gpd.sjoin(lights_map, grid[['cell_id', 'geometry']], predicate='intersects')
lights_in_grid = lights_in_grid[~lights_in_grid.index.duplicated()]

print('Crimes placed:', len(crime_in_grid))
print('Lights placed:', len(lights_in_grid))

`groupby('cell_id').size()` counts the points in each square. `map()` copies
those counts onto the grid; squares with no points get 0.

In [ ]:
for group in ['street', 'comparison']:
    counts = crime_in_grid[crime_in_grid['group'] == group].groupby('cell_id').size()
    grid[group + '_count'] = grid['cell_id'].map(counts).fillna(0)

grid['n_lights'] = grid['cell_id'].map(lights_in_grid.groupby('cell_id').size()).fillna(0)
grid['pct_led'] = grid['cell_id'].map(lights_in_grid.groupby('cell_id')['LED'].mean() * 100)

### Optional: commercial activity (OpenStreetMap)

Shops, pubs and takeaways show where people gather. If you have downloaded the
Geofabrik file, this cell counts them in each square.

In [ ]:
poi_file = Path('../data/raw/osm/gis_osm_pois_free_1.shp')
if poi_file.exists():
    pois = gpd.read_file(poi_file).to_crs(epsg=29902)
    commercial = ['pub', 'bar', 'nightclub', 'restaurant', 'fast_food', 'cafe',
                  'supermarket', 'convenience', 'department_store', 'mall', 'clothes']
    pois = pois[pois['fclass'].isin(commercial)]
    pois_in_grid = gpd.sjoin(pois, grid[['cell_id', 'geometry']], predicate='intersects')
    pois_in_grid = pois_in_grid[~pois_in_grid.index.duplicated()]
    grid['n_poi'] = grid['cell_id'].map(pois_in_grid.groupby('cell_id').size()).fillna(0)
else:
    print('No OpenStreetMap file found - skipping commercial activity.')

## Section 5 — Which squares are urban?

We use NISRA's **Settlement Development Limits (2015)**. Bands A–E are
settlements of 5,000 or more people, NISRA's default urban threshold.

`overlay(..., how='intersection')` cuts the grid squares into the parts that lie
inside a settlement. Each square is 1 km² (1,000,000 m²), so the area of the
pieces divided by 1,000,000 is the share of the square that is urban.

In [ ]:
settlements = gpd.read_file('../data/raw/settlements/settlements-2015-above-500-threshold.shp')
settlements = settlements.to_crs(epsg=29902)
urban_settlements = settlements[settlements['Band'].isin(['A', 'B', 'C', 'D', 'E'])]

pieces = gpd.overlay(grid[['cell_id', 'geometry']],
                     urban_settlements[['Name', 'Band', 'geometry']], how='intersection')
pieces['share'] = pieces.area / 1_000_000

grid['urban_share'] = grid['cell_id'].map(pieces.groupby('cell_id')['share'].sum()).fillna(0)
largest = pieces.sort_values('share').drop_duplicates('cell_id', keep='last').set_index('cell_id')
grid['settlement'] = grid['cell_id'].map(largest['Name'])

Our **study area** is squares that are at least half inside an urban settlement
and have at least 100 residents. Squares with very few residents (harbours,
industrial estates, parks) would give unstable crime rates per resident.

In [ ]:
grid['study_area'] = (grid['urban_share'] >= 0.5) & (grid['population'] >= 100)
study = grid[grid['study_area']].copy()
print('Study squares:', len(study))
print('Settlements:', study['settlement'].nunique())

In [ ]:
ax = grid.plot(color='lightgrey', figsize=(7, 7))
study.plot(ax=ax, color='tab:blue')
plt.title('Study area: urban 1 km squares')
plt.axis('off')
plt.show()

## Section 6 — Rates and the Street Light Availability Index (SLAI)

Divide counts by population to compare squares fairly. Rates are per 1,000 residents.

In [ ]:
for group in ['street', 'comparison']:
    study[group + '_rate'] = study[group + '_count'] / study['population'] * 1000

study['lights_per_km2'] = study['n_lights']          # every square is 1 km²
study['lights_per_1000'] = study['n_lights'] / study['population'] * 1000
study[['street_rate', 'comparison_rate', 'lights_per_km2', 'lights_per_1000']].describe()

The **SLAI** combines how densely a square is lit with how many lights there
are per resident. `rank(pct=True)` turns each measure into a percentile between
0 and 1, so a few very bright city-centre squares cannot dominate. The SLAI is
the average of the two percentiles.

We also split the squares into three equal groups (Low / Medium / High) with `qcut()`.

In [ ]:
study['slai'] = (study['lights_per_km2'].rank(pct=True) +
                 study['lights_per_1000'].rank(pct=True)) / 2
study['light_class'] = pd.qcut(study['slai'], 3, labels=['Low', 'Medium', 'High'])
study['light_class'].value_counts()

Finally, label each square with the **10 km block** it sits in. We will use the
blocks in notebook 03 to keep neighbouring squares together when testing models.

In [ ]:
study['x'] = study.geometry.centroid.x
study['y'] = study.geometry.centroid.y
study['block'] = (study['x'] // 10000).astype(int).astype(str) + '_' + \
                 (study['y'] // 10000).astype(int).astype(str)
study['block'].nunique()

In [ ]:
Path('../data/processed').mkdir(parents=True, exist_ok=True)
study['light_class'] = study['light_class'].astype(str)
study.to_file('../data/processed/study_area.gpkg')